# 03 · Estado dos verbetes antes e depois

A unidade desta etapa é `ano + turma + grupo + verbete`. O estado **Antes** vem do `parentid` da primeira revisão acadêmica da unidade. O **Depois** é a última revisão acadêmica identificada. Edições de outros usuários entre essas duas revisões podem afetar a comparação.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
BRUTOS = ROOT / 'dados' / 'brutos'
PROCESSADOS = ROOT / 'dados' / 'processados'
GRAFICOS = ROOT / 'resultados' / 'graficos'
PROCESSADOS.mkdir(parents=True, exist_ok=True)
GRAFICOS.mkdir(parents=True, exist_ok=True)

DET = pd.read_excel(PROCESSADOS/'resultado_tcc_contribuicoes_ampliado.xlsx',sheet_name='Detalhes_Edicoes')
PART = pd.read_excel(PROCESSADOS/'resultado_tcc_contribuicoes_ampliado.xlsx',sheet_name='Participacoes_Aluno')
FONTE = BRUTOS/'resultado_tcc_antes_depois_ampliado.xlsx'
REFERENCIA = pd.read_excel(FONTE, sheet_name='Antes_Depois_Ampliado')
WIKIDATA = pd.read_excel(FONTE, sheet_name='Interlinguisticos_Wikidata')

## Identificação das revisões de fronteira

In [ ]:
chave=['ano','turma','grupo','verbete']
det = DET.copy()
numeros = pd.to_numeric(det['timestamp'], errors='coerce')
como_excel = pd.to_datetime(numeros, unit='D', origin='1899-12-30', utc=True, errors='coerce')
como_texto = pd.to_datetime(det['timestamp'], utc=True, errors='coerce')
det['timestamp_ordem'] = como_excel.fillna(como_texto)
assert det['timestamp_ordem'].notna().all()
det = det.sort_values(['timestamp_ordem','revid'])
primeira = det.groupby(chave,dropna=False).first().reset_index()
ultima = det.groupby(chave,dropna=False).last().reset_index()
assert len(primeira)==len(ultima)==70
print('Unidades grupo/verbete:',len(primeira))

## Métricas dos dois estados

Reaproveitamos os valores de cada revisão já extraídos da API no notebook anterior. Para artigos novos, `parentid = 0` representa ausência de um estado anterior e as métricas anteriores são zero.

In [ ]:
metricas=['bytes_total','referencias','notas_rodape','secoes','infobox','ver_tambem',
          'links_ver_tambem','links_internos','links_externos','categorias','imagens','graficos','tabelas']
antes = primeira[chave+['parentid','revid','timestamp_ordem']+[f'{m}_antes' for m in metricas]].copy()
depois = ultima[chave+['revid','timestamp_ordem']+[f'{m}_depois' for m in metricas]].copy()
antes = antes.rename(columns={'parentid':'revid_antes','revid':'primeiro_revid','timestamp_ordem':'data_primeira_edicao'})
depois = depois.rename(columns={'revid':'revid_depois','timestamp_ordem':'data_ultima_edicao'})
unidades = antes.merge(depois,on=chave,validate='one_to_one')
unidades['criado_pelo_grupo'] = unidades['revid_antes'].fillna(0).eq(0).astype(int)
for met in metricas:
    unidades[f'{met}_variacao']=unidades[f'{met}_depois']-unidades[f'{met}_antes']

contagem = det.groupby(chave,dropna=False).size().reset_index(name='qtd_edicoes_grupo')
numero_integrantes = PART.groupby(chave,dropna=False)['id_participacao'].nunique().reset_index(name='qtd_integrantes_identificados')
unidades = unidades.merge(contagem,on=chave).merge(numero_integrantes,on=chave)

# A base histórica também contém uma identificação da unidade e metadados da atividade.
cols_meta=['id_unidade', *chave, 'tipo_participacao']
metadados = REFERENCIA[cols_meta].drop_duplicates(chave)
unidades = unidades.merge(metadados,on=chave,validate='one_to_one')
assert len(unidades)==70
assert int(unidades['criado_pelo_grupo'].sum())==21

## Validação histórica e exportação

A consulta direta a revisões pela API poderia ser reexecutada com a função `revisoes_por_id` do notebook 02. Nesta versão, utilizamos as revisões históricas e seus valores já preservados, evitando alterações de recorte numa coleta posterior.

In [ ]:
# Confere se as revisões-limite coincidem com a base final adotada no TCC.
ch = ['id_unidade']
val = unidades.merge(REFERENCIA[['id_unidade','revid_antes','revid_depois']],on=ch,suffixes=('','_hist'),validate='one_to_one')
assert (val['revid_antes'].fillna(0).astype(int)==val['revid_antes_hist'].fillna(0).astype(int)).all()
assert (val['revid_depois'].astype(int)==val['revid_depois_hist'].astype(int)).all()
for met in ['bytes_total','referencias','secoes','links_internos','imagens']:
    comp = unidades.set_index('id_unidade')[f'{met}_depois']
    original = REFERENCIA.set_index('id_unidade')[f'{met}_depois']
    assert comp.reindex(original.index).fillna(0).equals(original.fillna(0)), met

# Preserva também os campos complementares e a aba histórica de Wikidata.
saida=PROCESSADOS/'resultado_tcc_antes_depois_ampliado.xlsx'
with pd.ExcelWriter(saida,engine='openpyxl') as writer:
    REFERENCIA.to_excel(writer,sheet_name='Antes_Depois_Ampliado',index=False)
    WIKIDATA.to_excel(writer,sheet_name='Interlinguisticos_Wikidata',index=False)
    auditoria_exportar = unidades.copy()
    for coluna in ['data_primeira_edicao','data_ultima_edicao']:
        auditoria_exportar[coluna] = auditoria_exportar[coluna].dt.tz_localize(None)
    auditoria_exportar.to_excel(writer,sheet_name='Auditoria_Reconstrucao',index=False)
print('Arquivo gerado:',saida.name)